# RiskAI – Deep Learning mit TensorFlow & Keras

In dieser Phase bauen wir ein Künstliches Neuronales Netz (Artificial Neural Network) zur Erkennung von Kreditkartenbetrug.
Schritte:
1. Laden der skalierten und mit SMOTE ausbalancierten Trainingsdaten
2. Definition der Architektur des Neuronalen Netzes (Dense, BatchNormalization, Dropout, Sigmoid)
3. Training des Netzes mit Early Stopping (um Overfitting zu vermeiden)
4. Evaluierung und Vergleich mit unserem XGBoost-Champion


In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
import joblib


X_test = pd.read_csv('../data/processed/X_test_scaled.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()


X_train = pd.read_csv('../data/processed/X_train_scaled.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()

from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Daten erfolgreich geladen und resampelt!")
print(f"Trainingsset: {X_train_resampled.shape} | Testset: {X_test.shape}")
print(f"TensorFlow-Version: {tf.__version__}")


Daten erfolgreich geladen und resampelt!
Trainingsset: (453204, 32) | Testset: (56746, 32)
TensorFlow-Version: 2.21.0


In [24]:
model_nn = Sequential([

    Dense(64, activation="relu", input_shape=(X_train_resampled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),

    Dense(16, activation="relu"),
    BatchNormalization(),
    Dropout(0.2),

    Dense(1, activation="sigmoid")
])

model_nn.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_nn.summary()

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,185 (20.25 KB)

 Trainable params: 4,961 (19.38 KB)

 Non-trainable params: 224 (896.00 B)

In [25]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
restore_best_weights=True
)

history = model_nn.fit(
    X_train_resampled,  y_train_resampled,
    epochs=30,
    batch_size=2048,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9063 - loss: 0.2408 - val_accuracy: 0.9254 - val_loss: 0.1609
Epoch 2/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9666 - loss: 0.0941 - val_accuracy: 0.9794 - val_loss: 0.0602
Epoch 3/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9793 - loss: 0.0587 - val_accuracy: 0.9917 - val_loss: 0.0223
Epoch 4/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9867 - loss: 0.0388 - val_accuracy: 0.9988 - val_loss: 0.0082
Epoch 5/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9913 - loss: 0.0269 - val_accuracy: 0.9999 - val_loss: 0.0040
Epoch 6/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9935 - loss: 0.0209 - val_accuracy: 0.9998 - val_loss: 0.0058
Epoch 7/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9949 - loss: 0.0174 - val_accuracy: 1.0000 - val_loss: 0.0026
Epoch 8/30
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9959 - loss: 0.0144 - val_accuracy: 1.

## Evaluierung des Neuronalen Netzes

Wir testen das trainierte Modell nun auf den originalen, unbalancierten Testdaten.
Zuerst werten wir das Modell mit dem Standard-Schwellenwert (0.5) aus und optimieren danach die Entscheidungsschwelle.


In [26]:

y_probs_nn = model_nn.predict(X_test).ravel()
y_pred_nn_default = (y_probs_nn >= 0.5).astype(int)

print("=== NEURONALES NETZ (Standard-Threshold: 0.5) ===")
print(classification_report(y_test, y_pred_nn_default))
print("Konfusionsmatrix:")
print(confusion_matrix(y_test, y_pred_nn_default))


1774/1774 ━━━━━━━━━━━━━━━━━━━━ 0s 227us/step
=== NEURONALES NETZ (Standard-Threshold: 0.5) ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.76      0.79      0.77        95

    accuracy                           1.00     56746
   macro avg       0.88      0.89      0.89     56746
weighted avg       1.00      1.00      1.00     56746

Konfusionsmatrix:
[[56627    24]
 [   20    75]]


In [27]:
precisions_nn, recalls_nn, thresholds_nn = precision_recall_curve(y_test, y_probs_nn)


f1_scores_nn = 2 * (precisions_nn * recalls_nn) / (precisions_nn + recalls_nn + 1e-10)


ix_best_nn = np.argmax(f1_scores_nn)
best_threshold_nn = thresholds_nn[ix_best_nn] if ix_best_nn < len(thresholds_nn) else 0.5
best_f1_nn = f1_scores_nn[ix_best_nn]

print(f"Optimaler Threshold für das Neuronale Netz: {best_threshold_nn:.4f} (Max F1-Score: {best_f1_nn:.4f})")


Optimaler Threshold für das Neuronale Netz: 0.9990 (Max F1-Score: 0.8182)


In [28]:

y_pred_nn_opt = (y_probs_nn >= best_threshold_nn).astype(int)

print(f"=== NEURONALES NETZ (Optimierter Threshold: {best_threshold_nn:.4f}) ===")
print(classification_report(y_test, y_pred_nn_opt))
print("Konfusionsmatrix:")
print(confusion_matrix(y_test, y_pred_nn_opt))


=== NEURONALES NETZ (Optimierter Threshold: 0.9990) ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.89      0.76      0.82        95

    accuracy                           1.00     56746
   macro avg       0.94      0.88      0.91     56746
weighted avg       1.00      1.00      1.00     56746

Konfusionsmatrix:
[[56642     9]
 [   23    72]]


In [29]:

model_nn.save('../models/neural_network_smote.keras')

print("Neuronales Netz erfolgreich als 'neural_network_smote.keras' gespeichert!")


Neuronales Netz erfolgreich als 'neural_network_smote.keras' gespeichert!
